# Electrophysiology Analysis Notebook
**Author:** Carlos A. Guzman-Cruz  
**Version:** 2.0.1  

This notebook is the user-facing interface for patch-clamp data analysis.  
All UI logic lives in `patch_clamp_hub.py`; all math/DataFrame logic in `patch_clamp_analysis_helper.py`.

---
### Workflow
1. **Setup** – import hub & helper, verify installation.
2. **Data Scan** – find `.abf` files on disk.
3. **Select Files** – pick which files to work with.
4. **Simple Plots** – quick visualization of raw recordings.
5. **ISI / Analysis** – peak detection, ISI, and instant frequency.
6. **Save / Load** – persist results with joblib.


---
## 0 · Setup

In [1]:
# ── Imports ───────────────────────────────────────────────────────────────
import importlib
import patch_clamp_hub            as hub
import patch_clamp_analysis_helper as helper
import joblib
import pandas as pd
import numpy  as np

print(f'hub     v{hub.__version__}')
print(f'helper  v{helper.__version__}')
print('✅  All modules loaded successfully.')

hub     v2.0.1
helper  v2.0.1
✅  All modules loaded successfully.


### Hot-reload (run after editing .py files)
If you edit `patch_clamp_hub.py` or `patch_clamp_analysis_helper.py` during a session,
run the cell below to reload them without restarting the kernel.

In [2]:
importlib.reload(hub)
importlib.reload(helper)
print('🔄  Modules reloaded.')

🔄  Modules reloaded.


---
## 1 · Data Scan
Find `.abf` files on disk.  
- Choose **Scan Folder (recursive)** to find all `.abf` files under a directory tree.  
- Choose **Pick Individual File** to add a single file.  
- Press **Find Now** → **Save** to populate `hub.data_paths_found`.  
- Optionally rename keys (the file on disk is NOT renamed).  
- Press **Save to Memory** to persist as `paths_found.joblib`.

In [3]:
hub.data_scan()

In [ ]:
# Inspect the populated dictionary
hub.data_paths_found

---
## 2 · Select Files
Choose one or more files from `data_paths_found` for analysis.  
Hold **Ctrl / ⌘** to multi-select.  
Press **Save** to confirm → populates `hub.selected_file_names`.

In [ ]:
hub.select_files()

In [ ]:
# Confirm selection
print('Selected files:', hub.selected_file_names)

---
## 3 · Simple Plots
Quick visualization of raw recordings.  
1. Pick a file from the dropdown.  
2. Choose a plot type and rendering library (Matplotlib / Plotly).  
3. Press **Peak Qualities** to view metadata.  
4. Press **Plot** for a default full-range plot.  
5. Press **Clean Up** to reveal axis controls, then **Re-Plot**.  
6. Optionally tick **Download figure** and choose SVG or EPS.

In [ ]:
# min = -130
# max = 50
# thrhold = -22 

# Control                   0s - 8m 30s            0 - 510          
      # Mean ISI =            5.0888
      # Mean Inst. Freq =     0.5183

# Barium + Qinpirole    8m 30s - 34m             510 - 2040         
      # Mean ISI =           0.94187
      # Mean Inst. Freq =    1.75591

# Washout                  34m - 1hr 2m         2040 - 3720         
      # Mean ISI =           0.8058306  
      # Mean Inst. Freq =    1.4082565   

# Only Quinpirole       1hr 2m - end            3720 - 5333         
      # Mean ISI =          0.484571975  
      # Mean Inst. Freq =   2.2210002    

In [ ]:
29*60

In [ ]:
## MARCH 16th

# min = -130
# max = 50
# thrhold = -22 

#0000
# Control                   0s - 12m 40s            0 - 760          
      # Mean ISI =           14.2727568627451          
      # Mean Inst. Freq =     2.963573688036253  

# Barium + Qinpirole    12m 40s - 17m 55s           760 - 1075         
      # Mean ISI =            34.0817   
      # Mean Inst. Freq =     0.58632748   



#0001
# Barium + Qinpirole        0 - 17m 30s           0 - 1050         
      # Mean ISI =           13.153110
      # Mean Inst. Freq =    0.874151035

# Washout                17m 30s - 29m 30s         1050 - 1770         
      # Mean ISI =           42.341243
      # Mean Inst. Freq =    0.696063315   

# Only Quinpirole       29m 30s - 43m 31s             1770 - 2610         
      # Mean ISI =           5.1611  
      # Mean Inst. Freq =    1.99569   

In [ ]:
hub.simple_plots()

---
## 4 · Analysis via `patch_clamp_analysis_helper`

### 4a · Load a single sweep into a DataFrame

In [ ]:
# ── Choose a file key from hub.data_paths_found ───────────────────────────
FILE_KEY = list(hub.data_paths_found.keys())[2]   # change as needed
path     = hub.data_paths_found[FILE_KEY]

raw_df = helper.abf_to_dataframe(path, sweep=0)
print(f'Loaded: {FILE_KEY}  |  {len(raw_df):,} samples')
raw_df.head()

### 4b · Load all sweeps (multi-sweep protocols)

In [ ]:
all_sweeps_df = helper.abf_all_sweeps_to_dataframe(path)
print(f'Sweeps: {all_sweeps_df["sweep"].unique()}')
all_sweeps_df.head()

### 4c · Clean the DataFrame

In [ ]:
# ── Adjust these values to your recording ─────────────────────────────────
T_START      =   0.0    # seconds
T_END        = 500.0    # seconds
UPPER_THRESH =  50.0   # pA (drop artifacts above this)
LOWER_THRESH = -80.0   # pA (drop artifacts below this)

cleaned_df = helper.clean_dataframe(raw_df,
                                    t_start=T_START, t_end=T_END,
                                    upper_thresh=UPPER_THRESH,
                                    lower_thresh=LOWER_THRESH)
print(f'Cleaned: {len(cleaned_df):,} samples  ({len(raw_df) - len(cleaned_df):,} removed)')
cleaned_df.head()

### 4d · Detect Peaks / Spikes

In [ ]:
# ── Peak detection parameters ─────────────────────────────────────────────
SPIKE_THRESH    = -12.0    # pA threshold for spike detection
NEGATIVE_SPIKES = True   # True = downward (inward current) spikes
MIN_DEPTH_PA    = 0.0    # minimum extra depth above threshold
MIN_ISI_MS      = 0.0    # minimum inter-spike interval (0 = no filter)

sr = raw_df.attrs.get('sampleRate', 10_000.0)

peaks_df = helper.detect_peaks(cleaned_df,
                                threshold=SPIKE_THRESH,
                                negative_spikes=NEGATIVE_SPIKES,
                                min_depth=MIN_DEPTH_PA,
                                min_isi_ms=MIN_ISI_MS,
                                sample_rate=sr)
print(f'Detected {len(peaks_df)} spike(s).')
peaks_df.head(10)

### 4e · Compute ISI & Instant Frequency

In [ ]:
# =========================
# Plot VC trace + event peaks (red)
# =========================
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(12, 3.2), dpi=300)

# Main trace
ax.plot(
    cleaned_df["time"],
    cleaned_df["signal"],
    color="black",
    linewidth=0.5,
    zorder=1
)

# Overlay event peaks
ax.scatter(
    peaks_df["time"],
    peaks_df["signal"],
    color="red",
    s=12,            # marker size (points^2)
    zorder=3,
    label=f"Event peaks (n={len(peaks_df)})"
)

# Optional: threshold line
ax.axhline(SPIKE_THRESH, color="red", linewidth=1.0, linestyle="--", alpha=0.5)

# Labels / formatting
ax.set_title("VCGapFree (cell attached): Time vs Current (event peaks highlighted)")
ax.set_xlabel("Time (s)")
ax.set_ylabel("Current (pA)")

ymin_lim = cleaned_df["signal"].min() - 10
ymax_lim = cleaned_df["signal"].max() + 10

# Time of switch
#plt.axvline(480, color="red", linewidth=1.0, linestyle="--")

ax.set_ylim(ymin_lim, ymax_lim)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

ax.legend(frameon=False, loc="lower right")
plt.show()


In [ ]:
isi_df  = helper.compute_isi(peaks_df)
summary = helper.isi_summary(isi_df)

print('── ISI Summary ──────────────────────────────')
for k, v in summary.items():
    print(f'  {k:25s}: {v:.4f}' if isinstance(v, float) else f'  {k:25s}: {v}')

isi_df.head(10)

### 4f · Column utilities

In [ ]:
# Add rolling mean smoothing
cleaned_df = helper.add_rolling_mean(cleaned_df,
                                     window_ms=5.0,
                                     sample_rate=sr)

# Add z-score normalisation
cleaned_df = helper.add_zscore(cleaned_df)

# Add first-difference (dI/dt proxy)
cleaned_df = helper.add_delta_signal(cleaned_df)

print('Columns now:', list(cleaned_df.columns))
cleaned_df.head()

---
## 5 · Full Pipeline (shortcut)
Runs steps 4a–4e in one call and returns a dict ready to insert into `hub.isi_key_data`.

In [ ]:
result = helper.run_vc_pipeline(
    path          = path,
    t_start       = 0.0,
    t_end         = 200.0,
    upper_thresh  = 200.0,
    lower_thresh  = -200.0,
    spike_thresh  = 0.0,
    negative_spikes = True,
    min_depth     = 0.0,
    min_isi_ms    = 0.0,
)

# Store result in the global isi_key_data dictionary
hub.isi_key_data[FILE_KEY] = result

print('isi_key_data keys:', list(hub.isi_key_data.keys()))
print('Summary:')
for k, v in result['summary'].items():
    print(f'  {k:25s}: {v:.4f}' if isinstance(v, float) else f'  {k}: {v}')

---
## 6 · Save & Load Results

In [ ]:
# ── Save isi_key_data to disk ──────────────────────────────────────────────
joblib.dump(hub.isi_key_data, 'isi_key_data.joblib')
print('✅  Saved isi_key_data.joblib')

In [ ]:
# ── Load isi_key_data from disk ────────────────────────────────────────────
hub.isi_key_data = joblib.load('isi_key_data.joblib')
print(f'✅  Loaded {len(hub.isi_key_data)} entry(ies) from isi_key_data.joblib')

---
## 7 · Quick DataFrame Inspection Helpers

In [ ]:
# Inspect any stored DataFrame
key = list(hub.isi_key_data.keys())[0]
entry = hub.isi_key_data[key]

print(f'\n── cleaned_df for "{key}" ──────────────────')
display(entry['cleaned_df'].describe())

print(f'\n── peaks_df ───────────────────────────────')
display(entry['peaks_df'].head())

print(f'\n── isi_df ─────────────────────────────────')
display(entry['isi_df'].head())